# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 05: Mamba: Selective State Space Models from Scratch

### 1. Conceptual Intuition & Real-World Use Case
While S4 achieved linear scaling, it failed to match Transformer performance on natural language:
- In S4, the matrices $A, B, C$ are **Linear Time-Invariant (LTI)**: the model processes every token with the exact same weight transitions.
- But language requires **Content-Based Selection**: an LLM must remember critical tokens (e.g. key entity names) and selectively filter out filler words (e.g. "the", "uh").

In 2023, Albert Gu and Tri Dao introduced **Mamba (Selective State Spaces)**:
- Makes matrices $B, C$ and discretization step $\Delta$ **input-dependent functions of $x_t$**!
- Breaks LTI (can no longer be computed as a static convolution).
- Solves training parallelization using a **Hardware-Aware Associative Parallel Scan**!

---

### 2. Mathematical Foundations: The Selective Mechanism
Given input token vector $x_t \in \mathbb{R}^D$:

1. **Input-Dependent Projections:**
   $$B_t = \text{Linear}_B(x_t), \quad C_t = \text{Linear}_C(x_t), \quad \Delta_t = \text{Softplus}(\text{Parameter} + \text{Linear}_\Delta(x_t))$$
2. **Discretization:**
   $$\bar{A}_t = \exp(\Delta_t A), \quad \bar{B}_t = \Delta_t B_t$$
3. **Selective Recurrence:**
   $$h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t, \quad y_t = C_t h_t$$

When $\Delta_t \to 0$, $\bar{A}_t \to I$ and $\bar{B}_t \to 0$: the model ignores the current token and preserves past state.
When $\Delta_t \to \infty$, the model resets state and prioritizes current input!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Implement Mamba Selective SSM Layer
class MambaSelectiveSSM:
    def __init__(self, d_model=4, d_state=8):
        self.d_model = d_model
        self.d_state = d_state
        
        # Diagonal continuous A (d_model, d_state)
        # Initialized with negative values for stable exponential decay
        self.A = -np.random.uniform(0.5, 2.0, (d_model, d_state))
        
        # Linear projection weights to make B, C, Delta input-dependent
        self.W_B = np.random.randn(d_model, d_state) * 0.1
        self.W_C = np.random.randn(d_model, d_state) * 0.1
        self.W_delta = np.random.randn(d_model, d_model) * 0.1

    def forward_step(self, x_t, h_prev):
        """
        x_t: (d_model,)
        h_prev: (d_model, d_state)
        """
        # Step A: Compute input-dependent B_t, C_t, and Delta_t
        B_t = x_t @ self.W_B           # (d_state,)
        C_t = x_t @ self.W_C           # (d_state,)
        
        # Delta parameter (timescale gating)
        delta_raw = x_t @ self.W_delta
        delta_t = np.log1p(np.exp(delta_raw)) # Softplus activation
        
        # Step B: Discretize A and B (Zero-Order Hold)
        # A_bar = exp(delta_t * A)
        A_bar = np.exp(np.outer(delta_t, np.ones(self.d_state)) * self.A)
        # B_bar = delta_t * B
        B_bar = np.outer(delta_t, B_t)
        
        # Step C: State update: h_t = A_bar * h_{t-1} + B_bar * x_t
        h_t = A_bar * h_prev + B_bar * x_t[:, np.newaxis]
        
        # Step D: Output projection: y_t = sum_state (h_t * C_t)
        y_t = np.sum(h_t * C_t[np.newaxis, :], axis=1)
        
        return y_t, h_t, delta_t

# Step 2: Process a sequence of 4 tokens
d_model, d_state = 3, 4
mamba = MambaSelectiveSSM(d_model=d_model, d_state=d_state)

# Simulated token embeddings (e.g., [Alice, went, to, store])
tokens = np.random.randn(4, d_model)
h = np.zeros((d_model, d_state))

print("--- Mamba Selective Dynamic Step Progression ---")
for t in range(4):
    y_t, h, delta_t = mamba.forward_step(tokens[t], h)
    print(f"Token {t + 1}:")
    print(f"  Delta Gate mean: {np.mean(delta_t):.4f} (Dynamic time-step scaling)")
    print(f"  Output y_t:      {np.round(y_t, 3)}")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Discretization:** Notice $B_t, C_t, \Delta_t$ are dynamically derived from input $x_t$, allowing the network to modulate how much memory to retain.
2. **Linear Memory & Latency:** The hidden state $h_t$ maintains a fixed dimension $(D_{\text{model}} \times D_{\text{state}})$, eliminating the growing KV-cache memory bottleneck of Transformers.

---

### 4. Industry Applications & Trade-offs
- **Long-Context Language Modeling:** Mamba matches Transformer quality up to 7B parameters while generating tokens **5x faster** with zero memory scaling over 100K token sequences.
- **Associative Scan:** Training in parallel uses parallel prefix sums (associative scan), bypassing slow sequential loops.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Mamba discard the convolutional view of S4?**
> **Answer:** S4 achieves convolution because its parameters $A, B, C$ are static (Time-Invariant). In Mamba, $B_t, C_t, \Delta_t$ change at every single token step based on input $x_t$, breaking time-invariance. A static convolution kernel cannot represent dynamic input-dependent transitions.

**Q2: How does Mamba achieve $O(N)$ linear inference while Transformers require $O(N^2)$?**
> **Answer:** Transformers store all past Key and Value vectors in memory (KV-cache) and compute pairwise attention with all previous tokens. Mamba compresses all past context into a fixed-size latent state $h_t \in \mathbb{R}^{D \times N}$, computing updates in constant $O(1)$ time per step.